# 06 — Drift Simulation & Monitoring

Simulate temporal drift by splitting chronologically (train on early transactions, test on later ones) instead of randomly, then measure feature-level drift using PSI and Evidently AI reports. This addresses a key production concern: fraud patterns change over time, and models need active monitoring to detect when they've become stale.

## 1. Load Raw Data and Sort by Time

Reload the cleaned (deduped, dtype-fixed) dataset and sort by the `Time` column to enable a genuine chronological split, rather than the random stratified split used in Layers 1-4.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/creditcard.csv')
df['Class'] = df['Class'].astype(str).str.strip("'").astype(int)
df = df.drop_duplicates()

df = df.sort_values('Time').reset_index(drop=True)

print(df.shape)
print("Time range:", df['Time'].min(), "to", df['Time'].max())
print(df['Class'].value_counts())

(283726, 31)
Time range: 0.0 to 172792.0
Class
0    283253
1       473
Name: count, dtype: int64


## 2. Chronological Split — Early vs Late Transactions

Split the data by time rather than randomly: the first ~70% of transactions (chronologically) become the "training" reference period, and the last ~30% become the "recent" period we'll monitor for drift. This simulates a realistic production scenario — a model trained on historical data, evaluated against more recent incoming transactions.

In [2]:
split_point = int(len(df) * 0.7)

df_early = df.iloc[:split_point].copy()
df_late = df.iloc[split_point:].copy()

print("Early period (reference/training):", df_early.shape, "-", df_early['Class'].sum(), "fraud")
print("Late period (recent/monitoring):", df_late.shape, "-", df_late['Class'].sum(), "fraud")

print("\nEarly period time range:", df_early['Time'].min(), "-", df_early['Time'].max())
print("Late period time range:", df_late['Time'].min(), "-", df_late['Time'].max())

Early period (reference/training): (198608, 31) - 366 fraud
Late period (recent/monitoring): (85118, 31) - 107 fraud

Early period time range: 0.0 - 132906.0
Late period time range: 132906.0 - 172792.0


## 3. Train a Model on the Early Period, Evaluate on the Late Period

Train XGBoost using only early-period data, then evaluate it on late-period data — this measures whether a model trained on historical patterns degrades when applied to more recent transactions, simulating realistic model staleness in production.

In [3]:
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

# Scale Amount/Time using ONLY early-period statistics (as would happen in real production)
scaler = StandardScaler()
df_early[['Amount', 'Time']] = scaler.fit_transform(df_early[['Amount', 'Time']])
df_late[['Amount', 'Time']] = scaler.transform(df_late[['Amount', 'Time']])  # transform only, no fit

X_early, y_early = df_early.drop(columns=['Class']), df_early['Class']
X_late, y_late = df_late.drop(columns=['Class']), df_late['Class']

scale_pos_weight_temporal = (y_early == 0).sum() / (y_early == 1).sum()

temporal_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight_temporal,
    eval_metric='aucpr',
    random_state=42
)
temporal_model.fit(X_early, y_early)

late_proba = temporal_model.predict_proba(X_late)[:, 1]
temporal_pr_auc = average_precision_score(y_late, late_proba)

print(f"PR-AUC on late (unseen future) period: {temporal_pr_auc:.4f}")
print(f"Compare to original random-split PR-AUC: 0.8568")

PR-AUC on late (unseen future) period: 0.7929
Compare to original random-split PR-AUC: 0.8568


## 4. Population Stability Index (PSI) — Feature-Level Drift

Calculate PSI for each feature to identify exactly which ones shifted most between the early (training) and late (monitoring) periods. PSI is a standard industry metric: values below 0.1 indicate no significant shift, 0.1–0.25 indicate moderate shift, and above 0.25 indicates significant shift requiring attention.

In [6]:
def calculate_psi(expected, actual, bins=10):
    """Calculate PSI between two distributions using quantile-based bins from the expected (reference) distribution."""
    breakpoints = np.quantile(expected, np.linspace(0, 1, bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    expected_bins = pd.Series(pd.cut(expected, bins=breakpoints, duplicates='drop'))
    actual_bins = pd.Series(pd.cut(actual, bins=breakpoints, duplicates='drop'))

    expected_counts = expected_bins.value_counts()
    actual_counts = actual_bins.value_counts()

    expected_pct = expected_counts / expected_counts.sum()
    actual_pct = actual_counts / actual_counts.sum()

    # align indices in case some bins are missing in one distribution
    expected_pct, actual_pct = expected_pct.align(actual_pct, fill_value=0.0001)

    # avoid division by zero / log(0)
    expected_pct = expected_pct.replace(0, 0.0001)
    actual_pct = actual_pct.replace(0, 0.0001)

    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

psi_results = {}
for col in X_early.columns:
    psi_results[col] = calculate_psi(X_early[col].values, X_late[col].values)

psi_df = pd.Series(psi_results).sort_values(ascending=False)
print(psi_df)

Time      8.283070
V1        0.972458
V3        0.764662
V28       0.532773
V11       0.333583
V25       0.249938
V15       0.212968
V12       0.202600
V22       0.164561
V5        0.158365
V4        0.125207
V24       0.123715
V23       0.123401
V26       0.121969
V21       0.103651
V7        0.075855
V14       0.070492
V6        0.059956
V9        0.054934
V20       0.053843
V8        0.048872
V13       0.042543
V27       0.042455
V17       0.037145
V10       0.032706
V19       0.024616
V18       0.024063
V2        0.016271
V16       0.005266
Amount    0.004237
dtype: float64


### Note on `Time` PSI

`Time` shows an extremely high PSI (8.28) purely because we performed a chronological split — by construction, late-period `Time` values fall entirely outside the early period's range. This is not meaningful drift and is excluded from interpretation below. The feature-level drift analysis focuses on `V1`–`V28` and `Amount`.